In [1]:
import numpy as np
import matplotlib.pyplot as plt
import GCN
import util

#adjacencyList = np.load("filtered_data/hseAdjacencyList.npy", allow_pickle=True)

TYPE = ""
TYPE2 = "hse"

inputFeatureVectorList = np.load(f"filtered_data/input{TYPE}FeatureVector.npy").astype(np.double)
chargeStateList = np.load(f"filtered_data/defect{TYPE}ChargeState.npy", allow_pickle=True)
outputLabelList = np.load(f"filtered_data/output{TYPE}FeatureVector.npy", allow_pickle=True)
adjMatrix = np.load(f"filtered_data/{TYPE2}{TYPE}AdjacencyMatrix.npy", allow_pickle=True)
defectElementList = np.load(f'filtered_data/elementsFilteredListNoPadding.npy', allow_pickle=True)

MAX_NODES = 15

inputFeatureVectorList = inputFeatureVectorList[:, :MAX_NODES, :]
adjMatrix = adjMatrix[:, :MAX_NODES, :MAX_NODES]
#defectElementList = np.array([list(el)[:MAX_NODES] for el in defectElementList], dtype=object)

# Keep distances for edges below threshold, set others to 0 (no edge)
DISTANCE_THRESHOLD = .25
adjMatrix = np.where(adjMatrix < DISTANCE_THRESHOLD, (DISTANCE_THRESHOLD - adjMatrix)/DISTANCE_THRESHOLD, 0)





In [ ]:
import random

length = util.getPeriodicLength()

seed = 42
np.random.seed(seed)
random.seed(seed)

# Initialize with small random values instead of large negative numbers
# This gives a better starting point for optimization
alpha_params = np.random.rand(length) * 2 - 1  # Random values between -1 and 1
beta_params = np.random.rand(length, length) * 0.2 - 0.1  # Random values between -0.1 and 0.1


def build_model(idx, input_shape):
    # input_shape is a tuple like (num_samples, num_atoms, num_features)
    # We want num_atoms, which is input_shape[1]
    num_atoms = input_shape[1]
    # Initialize Hamiltonian matrix
    H = np.zeros((num_atoms, num_atoms))
    # Add on-site energies (alpha parameters) to diagonal
    for i in range(num_atoms):
        # Get element type for atom i from input features
        element_type = np.argmax(inputFeatureVectorList[idx][i])  # Assuming one-hot encoded
        H[i, i] = alpha_params[element_type]
    
    # Add hopping terms (beta parameters) for connected atoms
    for i in range(num_atoms):
        for j in range(num_atoms):
            if i != j and adjMatrix[idx][i, j] > 0:  # If atoms are connected
                element_i = np.argmax(inputFeatureVectorList[idx][i])
                element_j = np.argmax(inputFeatureVectorList[idx][j])
                # Hopping parameter weighted by adjacency (distance-based weight)
                H[i, j] = beta_params[element_i, element_j] * adjMatrix[idx][i, j]

    return H

def compute_model_output(H):
    # Compute eigenvalues of the Hamiltonian matrix
    eigenvalues, eigenvectors = np.linalg.eigh(H)

    sorted_labels = eigenvalues[np.argmax(abs(eigenvectors), axis=0)]

    return sorted_labels 

def compute_MSE(idx):
    H = build_model(idx, inputFeatureVectorList.shape)
    sorted_labels = compute_model_output(H)
    # Use idx instead of undefined sample_idx
    defect_energy = np.sum(sorted_labels[:len(defectElementList[idx])])
    MSE = (defect_energy - outputLabelList[idx]["hse-gs"]["defect_total_energy"])**2
    return MSE


In [ ]:
def train_step(idx, learning_rate=0.01):
    # Compute gradients numerically
    epsilon = 1e-6
    max_gradient = 10.0  # Clip gradients to prevent instability

    original_MSE = compute_MSE(idx)
    
    non_OHE = np.argmax(inputFeatureVectorList[idx], axis=1)

    # Get unique element types present in this sample
    element_types_in_sample = set()
    for i in range(inputFeatureVectorList.shape[1]):  # MAX_NODES
        element_type = non_OHE[i]
        element_types_in_sample.add(element_type)
    
    # Update only alpha parameters for elements present in this sample
    for i in element_types_in_sample:
        alpha_params[i] += epsilon
        new_MSE = compute_MSE(idx)
        gradient = (new_MSE - original_MSE) / epsilon
        # Clip gradient to prevent instability
        gradient = np.clip(gradient, -max_gradient, max_gradient)
        alpha_params[i] -= epsilon  # Reset parameter
        alpha_params[i] -= learning_rate * gradient  # Gradient descent step

    # Update only beta parameters for element pairs present in this sample
    # Get element pairs that are connected in this sample
    element_pairs = set()
    for i in range(inputFeatureVectorList.shape[1]):
        for j in range(inputFeatureVectorList.shape[1]):
            if i != j and adjMatrix[idx][i, j] > 0:  # If atoms are connected
                element_i = non_OHE[i]
                element_j = non_OHE[j]
                element_pairs.add((element_i, element_j))
    
    for (i, j) in element_pairs:
        beta_params[i, j] += epsilon
        new_MSE = compute_MSE(idx)  # Fixed: removed extra argument
        gradient = (new_MSE - original_MSE) / epsilon
        # Clip gradient to prevent instability
        gradient = np.clip(gradient, -max_gradient, max_gradient)
        beta_params[i, j] -= epsilon  # Reset parameter
        beta_params[i, j] -= learning_rate * gradient  # Gradient descent step

def train_epochs(num_epochs=10, learning_rate=0.01):
    for epoch in range(num_epochs):
        total_MSE = 0
        for idx in range(len(inputFeatureVectorList)):
            train_step(idx, learning_rate)
            total_MSE += compute_MSE(idx)  # Fixed: removed extra argument
        avg_MSE = total_MSE / len(inputFeatureVectorList)
        print(f"Epoch {epoch+1}/{num_epochs}, Average MSE: {avg_MSE}")
        
        # Clip parameters to prevent them from growing too large
        np.clip(alpha_params, -10, 10, out=alpha_params)
        np.clip(beta_params, -10, 10, out=beta_params)

# Use a much smaller learning rate given the large MSE values
train_epochs(num_epochs=5, learning_rate=0.001)
print(f"Final alpha params (non-zero): {alpha_params[alpha_params != 0][:10]}")
print(f"Final beta params (sample): {beta_params[0, :10]}")

Epoch 1/5, Average MSE: 174345670568.473
Epoch 2/5, Average MSE: 174321027407.50797
Epoch 2/5, Average MSE: 174321027407.50797
Epoch 3/5, Average MSE: 174321027358.37564
Epoch 3/5, Average MSE: 174321027358.37564
Epoch 4/5, Average MSE: 174321027406.0777
Epoch 4/5, Average MSE: 174321027406.0777
Epoch 5/5, Average MSE: 174321027925.7998
Final alpha params (non-zero): [-10.         -10.          10.          10.         -10.
 -10.          -8.07746371   6.17281499  10.          10.        ]
Final beta params (sample): [-10. -10. -10. -10. -10. -10. -10. -10. -10. -10.]
Epoch 5/5, Average MSE: 174321027925.7998
Final alpha params (non-zero): [-10.         -10.          10.          10.         -10.
 -10.          -8.07746371   6.17281499  10.          10.        ]
Final beta params (sample): [-10. -10. -10. -10. -10. -10. -10. -10. -10. -10.]
